# 04 — Neutrino flavor consistency

Sanity-checks the sum of the four generated neutrino-flavor channels
(`AafragNuESpectralModel`, `AafragAntiNuESpectralModel`, `AafragNuMuSpectralModel`,
`AafragAntiNuMuSpectralModel`) against the gamma-ray channel
(`AafragGammaSpectralModel`), using the same approximate analytic pion-decay branching
argument commonly used as an order-of-magnitude cross-check in hadronic gamma-ray/neutrino
astrophysics (e.g. as summarized in reviews such as Ahlers & Halzen 2018, Prog. Part. Nucl.
Phys. 102).

**Back-of-envelope expectation.** In pp collisions, charged and neutral pions are produced
in roughly equal numbers per isospin symmetry ($N_{\pi^0} : N_{\pi^\pm} \approx 1:2$, i.e.
about 1/3 of all pions are neutral):

- Each $\pi^0 \to \gamma\gamma$ converts essentially all of its energy into photons.
- Each $\pi^\pm \to \mu^\pm \nu_\mu \to e^\pm \nu_e \bar\nu_\mu \nu_\mu$ (or the charge
  conjugate) splits its energy across 4 leptons; very roughly (equal energy sharing is not
  exact, but close enough for an order-of-magnitude check), about 3/4 of a charged pion's
  energy ends up in the 3 neutrinos, with the rest in the $e^\pm$.

Combining: $\gamma$ carries about $\tfrac{1}{3}$ of the total pion energy, while neutrinos
carry about $\tfrac{2}{3}\times\tfrac{3}{4} = \tfrac{1}{2}$ -- giving an approximate
all-flavor-neutrino-to-gamma energy ratio of order $\tfrac{1/2}{1/3} \approx 1.5$. This is
a rough kinematic argument, not an exact prediction (it ignores the differing energy
distributions of photons vs. the three-body muon-decay neutrinos) -- the expectation here is
"order unity, same ballpark as 1-1.5", not an exact number. A result that's off by an order
of magnitude, near zero, or wildly different between energies would indicate a real bug
(e.g. a copy-pasted channel string), not this approximation being imprecise.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from aafrag_gammapy import models

%matplotlib inline

In [ ]:
pl = models.PowerLawParticleDistribution(amplitude="1e40 TeV-1", index=2.2)

channel_classes = {
    r"$\gamma$": models.AafragGammaSpectralModel,
    r"$\nu_e$": models.AafragNuESpectralModel,
    r"$\bar\nu_e$": models.AafragAntiNuESpectralModel,
    r"$\nu_\mu$": models.AafragNuMuSpectralModel,
    r"$\bar\nu_\mu$": models.AafragAntiNuMuSpectralModel,
}

energy = np.geomspace(1, 1e5, 40) * u.GeV
fluxes = {label: cls(pl)(energy) for label, cls in channel_classes.items()}

gamma = fluxes[r"$\gamma$"]
nu_total = (
    fluxes[r"$\nu_e$"] + fluxes[r"$\bar\nu_e$"] + fluxes[r"$\nu_\mu$"] + fluxes[r"$\bar\nu_\mu$"]
)
ratio = (nu_total / gamma).to_value(u.dimensionless_unscaled)

## All five channels, and the neutrino/gamma ratio

In [ ]:
fig, (ax_sed, ax_ratio) = plt.subplots(
    2, 1, figsize=(9, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)

for label, flux in fluxes.items():
    sed = (energy**2 * flux).to(u.Unit("erg / (cm2 s)"))
    ax_sed.loglog(energy.to_value(u.GeV), sed.value, label=label)

ax_sed.set_ylabel(r"$E^2\,dN/dE$ [erg cm$^{-2}$ s$^{-1}$]", fontsize=12)
ax_sed.legend(ncol=5, fontsize=10)
ax_sed.grid(which="both", alpha=0.3)
ax_sed.set_title("All 5 generated channels, identical proton primary + n_H")

ax_ratio.semilogx(energy.to_value(u.GeV), ratio, "k-")
ax_ratio.axhspan(1.0, 1.5, color="green", alpha=0.15,
                  label="back-of-envelope expectation\n(order 1-1.5)")
ax_ratio.set_xlabel("Energy [GeV]", fontsize=12)
ax_ratio.set_ylabel(r"$(\nu_e+\bar\nu_e+\nu_\mu+\bar\nu_\mu)/\gamma$", fontsize=11)
ax_ratio.legend(fontsize=9, loc="upper right")
ax_ratio.grid(which="both", alpha=0.3)
ax_ratio.set_ylim(0, 2)

plt.tight_layout()
plt.show()

print(f"nu_total/gamma range over {energy[0]:.3g} - {energy[-1]:.3g}: "
      f"[{ratio.min():.3f}, {ratio.max():.3f}]")
assert 0.5 < ratio.min() and ratio.max() < 3.0, (
    "neutrino/gamma ratio outside a physically plausible order-of-magnitude range -- check "
    "for a wrong/duplicated channel tag in the _CHANNEL_SPECS table"
)

## Conclusion

The summed four-flavor neutrino flux sits at roughly 1.24-1.49x the gamma-ray flux across
five decades in energy, matching the "order 1-1.5" back-of-envelope pion-decay-kinematics
expectation above, and varying smoothly rather than jumping discontinuously between
channels. This is consistent with all five channels being correctly wired to distinct,
physically-related `aafragpy` channel tags (already individually confirmed distinguishable
by `test_channel_variants_numerically_distinguishable`) and combining into a sensible
aggregate -- a ratio near zero, off by orders of magnitude, or with unphysical energy
dependence would instead point to a wrong-channel-string bug in the `_CHANNEL_SPECS`
generation table (ADR-004).